In [2]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Configurar y registrar la ruta principal del proyecto
import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/TFM_education_ai_analytics")
TRANSFORMERS_PATH = PROJECT_ROOT / "educational_ai_analytics" / "2_modeling" / "transformers"

# Añadimos la carpeta directamente para evitar el módulo con nombre numérico
if str(TRANSFORMERS_PATH) not in sys.path:
    sys.path.insert(0, str(TRANSFORMERS_PATH))

# Ahora el import es plano y sin el `2_modeling` problemático
from transformer_GLU_classifier import GLUTransformerClassifier
from train_transformer import load_and_prepare_split


# -------------------------------------------------------------
# 2. Búsqueda y Extracción Dinámica de Hiperparámetros
# -------------------------------------------------------------
TARGET_TIMESTAMP = "2026-03-03 09:53:44" # <-- CLAVE ÚNICA DEL EXPERIMENTO

# Seteamos aquí el bin (Week) y target base que usaste para esa fecha
UPTO_WEEK = 5
NUM_CLASSES = 2
BINARY_MODE = "paper"
PAPER_BASELINE = True

HISTORY_PATH = PROJECT_ROOT / "reports" / "transformer_training" / f"week_{UPTO_WEEK}" / f"experiments_history_{NUM_CLASSES}clases_{BINARY_MODE}.json"
MODEL_WEIGHTS_PATH = PROJECT_ROOT / "models" / "transformers" / f"transformer_uptoW{UPTO_WEEK}_{NUM_CLASSES}clases_{BINARY_MODE}.keras"

print(f"🔍 Buscando hiperparámetros del experimento: {TARGET_TIMESTAMP}...")
with open(HISTORY_PATH, "r") as f:
    history = json.load(f)

# Filtrado por Timestamp
experiment = next((exp for exp in history if exp["timestamp"] == TARGET_TIMESTAMP), None)
if experiment is None:
    raise ValueError(f"No se encontró el experimento con timestamp {TARGET_TIMESTAMP}")

hp = experiment["hyperparameters"]
WITH_STATIC = hp.get("with_static", True)
print("✅ Hiperparámetros recuperados con éxito:")
for k, v in hp.items():
    print(f"  - {k}: {v}")

# -------------------------------------------------------------
# 3. Cargar Conjunto de Datos de Validación para el XAI
# -------------------------------------------------------------
BASE_NPZ_PATH = PROJECT_ROOT / "data" / "6_transformer_features"

X_seq_val, mask_pad_val, mask_activity_val, X_static_val, y_val, ids_val, cluster_dim_val, static_names_val, static_sources_val = load_and_prepare_split(
    base_npz=BASE_NPZ_PATH,
    split="validation", 
    w_key=UPTO_WEEK,
    num_classes=NUM_CLASSES,
    paper_baseline=PAPER_BASELINE,
    with_static=WITH_STATIC,
    binary_mode=BINARY_MODE
)

print(f"\n📊 Datos XAI cargados -> Secuencias: {X_seq_val.shape} | Estáticas: {X_static_val.shape if X_static_val is not None else 'N/A'}")

val_inputs = [X_seq_val.astype(np.float32), mask_pad_val.astype(np.int32), mask_activity_val.astype(np.int32)]
if WITH_STATIC and X_static_val is not None:
    val_inputs.append(X_static_val.astype(np.float32))

# -------------------------------------------------------------
# 4. Construcción Estructural Dinámica y Cargar Pesos
# -------------------------------------------------------------
print("\n🏗️ Levantando topología de la red (GLU Transformer Classifier) parametrizada dinámicamente...")
model = GLUTransformerClassifier(
    latent_d=hp["latent_d"],
    num_heads=hp["num_heads"],
    ff_dim=hp["ff_dim"],
    dropout=0.0, # IMPORTANTE: Reescrito a 0.0 para determinismo en el motor de explicabilidad (SHAP no oscila)
    num_classes=NUM_CLASSES,
    num_layers=hp["num_layers"],
    max_len=X_seq_val.shape[1],
    with_static_features=(WITH_STATIC and X_static_val is not None),
    static_hidden=hp["static_hidden"],
    head_hidden=hp["head_hidden"]
)

# Inicializar tensores (Dummy Batch)
dummy_inputs = [inp[:2] for inp in val_inputs]
_ = model(dummy_inputs, training=False)

# Keras permite inyectar una estructura salvada como .keras íntegro a un esqueleto vivo haciendo load_weights()
model.load_weights(MODEL_WEIGHTS_PATH)
print(f"✅ Pesos restaurados sin error desde {MODEL_WEIGHTS_PATH}")

print("\n🚀 Sistema XAI completamente preparado.")


2026-03-03 10:29:33.543823: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.11/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr

🔍 Buscando hiperparámetros del experimento: 2026-03-03 09:53:44...
✅ Hiperparámetros recuperados con éxito:
  - upto_week: 5
  - num_classes: 2
  - paper_baseline: True
  - binary_mode: paper
  - batch_size: 64
  - epochs: 80
  - with_static: True
  - use_clustering_features: True
  - accumulated_uptow: True
  - latent_d: 256
  - num_heads: 8
  - ff_dim: 128
  - dropout: 0.3
  - num_layers: 2
  - static_hidden: [64, 32]
  - head_hidden: [256, 128, 64, 32]
  - learning_rate: 0.001
  - clipnorm: 1.0
  - reduce_lr_factor: 0.2
  - reduce_lr_patience: 7
  - reduce_lr_min_lr: 1e-06
  - early_stopping_patience: 10
  - focal_gamma: 2.5
  - focal_alpha: [0.27, 0.73]
2026-03-03 10:29:39.660 | INFO     | train_transformer:filter_classes:78 - Configuración 2 clases (Baseline Paper): [0: Pass/Dist] vs [1: Withdrawn]. (Fail eliminado)

📊 Datos XAI cargados -> Secuencias: (3252, 5, 20) | Estáticas: (3252, 65)

🏗️ Levantando topología de la red (GLU Transformer Classifier) parametrizada dinámicamente.

I0000 00:00:1772533780.682623   49557 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5580 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


✅ Pesos restaurados sin error desde /workspace/TFM_education_ai_analytics/models/transformers/transformer_uptoW5_2clases_paper.keras

🚀 Sistema XAI completamente preparado.
